# PROYECTO FINAL: APLICACIONES ANALÍTICAS DE BIG DATA (UAPA)
## Auditoría de Experiencia y Sentimiento de Marca en Telecomunicaciones vía YouTube Data API v3
### Caso de Estudio: Claro República Dominicana (@clarord)

**Facilitador:** Luis Eduardo Bayonet Robles  
**Integrantes del Equipo:**
- **Audric André Rosario Rosario** (Matrícula: 100089140) — *Lead Data Engineering & NLP Modeling*
- **Orlando Benítez Ventura** (Matrícula: 100090873) — *Lead Business Intelligence & Executive Strategy*

---
### Objetivos del Notebook:
1. Demostrar la extracción ética y prudente de comentarios reales usando **YouTube Data API v3**.
2. Ejecutar el preprocesamiento lingüístico del español dominicano.
3. Implementar un pipeline de **NLP con arquitectura Transformer preentrenada** para clasificación de sentimiento y detección de tópicos.
4. Calcular el **Net Sentiment Score (NSS)** e indicadores de gestión empresarial.

### 1. Carga de Librerías y Configuración del Entorno

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Añadir src al path para reutilizar módulos modulares
sys.path.append(os.path.abspath('../src'))
from preprocesamiento import preprocesar_dataframe, limpiar_texto
from modelado_nlp import SentimentTransformerPipeline

print("Entorno inicializado correctamente.")
print("Librerías cargadas: Pandas, NumPy, Plotly, Transformers, PyTorch.")

Entorno inicializado correctamente.
Librerías cargadas: Pandas, NumPy, Plotly, Transformers, PyTorch.


### 2. Carga y Exploración de Datos Crudos de YouTube (Claro RD)

In [2]:
raw_path = '../data/raw/youtube_claro_raw.csv'
df_raw = pd.read_csv(raw_path)
print(f"Total de registros cargados: {len(df_raw)}")
print("Videos analizados: 56 videos corporativos y comparativas")
print("Primeras 5 observaciones extraídas de YouTube Data API v3:\n")
df_raw[['video_title', 'author', 'comment_text', 'published_at', 'like_count']].head()

Total de registros cargados: 799
Videos analizados: 56 videos corporativos y comparativas
Primeras 5 observaciones extraídas de YouTube Data API v3:

                                                     video_title                author                                                                                                               comment_text          published_at  like_count
0                             Claro RD - La Conexión que nos une  @esperanzaraposo9198                                                                                       Pueden aser.esa publicidad más corta  2026-08-30T19:05:27Z           0
1  0800 CLARO 🔴 Número de Teléfono de Atención al Cliente (2026)           @donuba1945  Me obligan a poner recarga amenazándome con eliminar la línea soy Ubaldo Gutiérrez Cadena de tercera edad y discapacitado  2026-08-26T16:54:27Z           0
2  0800 CLARO 🔴 Número de Teléfono de Atención al Cliente (2026)    @JuniorDelgado-e2s                                

### 3. Pipeline de Preprocesamiento y Limpieza de Texto en Español

In [3]:
df_clean = preprocesar_dataframe(df_raw, columna_texto='comment_text')
print("Muestra de texto original vs texto limpio y normalizado:\n")
df_clean[['comment_text', 'clean_text', 'service_category' if 'service_category' in df_clean.columns else 'topic_category']].head(5)

Muestra de texto original vs texto limpio y normalizado:

                                                                                                                comment_text                                                                                                                    clean_text              service_category
0                                                                                       Pueden aser.esa publicidad más corta                                                                                         pueden aser esa publicidad ma s corta  Experiencia General de Marca
1  Me obligan a poner recarga amenazándome con eliminar la línea soy Ubaldo Gutiérrez Cadena de tercera edad y discapacitado  me obligan a poner recarga amenaza ndome con eliminar la li nea soy ubaldo gutie rrez cadena de tercera edad y discapacitado         Facturación y Tarifas
2                                                                                                  

### 4. Clasificación de Sentimiento con Modelo Transformer Preentrenado

In [4]:
pipeline_sentimiento = SentimentTransformerPipeline()
df_scored = pipeline_sentimiento.procesar_dataframe(df_clean, columna_texto='clean_text')
print(f"[NLP] Inferencia completada para {len(df_scored)} comentarios.")
df_scored[['clean_text', 'sentiment_label', 'sentiment_score']].head(8)

[NLP] Inferencia completada para 799 comentarios.
Muestra de comentarios con etiquetas predichas y puntaje de confianza (score):

                                                                                                                                                                                                                                                                               clean_text sentiment_label  sentiment_score
0                                                                                                                                                                                                                                                   pueden aser esa publicidad ma s corta          NEUTRO             0.60
1                                                                                                                                                            me obligan a poner recarga amenaza ndome con eliminar la li nea soy ubaldo gutie rr

### 5. Cálculo del Net Sentiment Score (NSS) y Métricas Ejecutivas

In [5]:
total = len(df_scored)
dist_sent = df_scored['sentiment_label'].value_counts()
pct_pos = (dist_sent.get('POSITIVO', 0) / total) * 100
pct_neg = (dist_sent.get('NEGATIVO', 0) / total) * 100
pct_neu = (dist_sent.get('NEUTRO', 0) / total) * 100
nss = pct_pos - pct_neg

print("=======================================================")
print("--- MÉTRICAS GERENCIALES CLARO DOMINICANA ---")
print(f"Volumen total auditado: {total} interacciones")
print(f"Comentarios Positivos:  {dist_sent.get('POSITIVO', 0)} ({pct_pos:.2f}%)")
print(f"Comentarios Negativos:  {dist_sent.get('NEGATIVO', 0)} ({pct_neg:.2f}%)")
print(f"Comentarios Neutros:    {dist_sent.get('NEUTRO', 0)} ({pct_neu:.2f}%)")
print(f"Net Sentiment Score (NSS / NPS Estimado): {nss:+.2f}%")
print("=======================================================")

--- MÉTRICAS GERENCIALES CLARO DOMINICANA ---
Volumen total auditado: 799 interacciones
Comentarios Positivos:  99 (12.39%)
Comentarios Negativos:  53 (6.63%)
Comentarios Neutros:    647 (80.98%)
Net Sentiment Score (NSS / NPS Estimado): +5.76%


### 6. Visualización Interactiva con Plotly

In [6]:
fig_pie = px.pie(
    df_scored,
    names='sentiment_label',
    color='sentiment_label',
    color_discrete_map={'POSITIVO': '#2ECC71', 'NEGATIVO': '#E74C3C', 'NEUTRO': '#95A5A6'},
    hole=0.5,
    title="Distribución de Sentimiento Global de Claro Dominicana en YouTube"
)
fig_pie.show()

<Figure: Distribución de Sentimiento Global de Claro Dominicana en YouTube>

In [7]:
cat_summary = df_scored.groupby([cat_col, 'sentiment_label']).size().reset_index(name='conteo')
fig_bar = px.bar(
    cat_summary,
    x=cat_col,
    y='conteo',
    color='sentiment_label',
    barmode='group',
    color_discrete_map={'POSITIVO': '#2ECC71', 'NEGATIVO': '#E74C3C', 'NEUTRO': '#95A5A6'},
    title="Sentimiento de Marca por Categoría de Servicio (Claro RD)",
    labels={cat_col: 'Servicio', 'conteo': 'Volumen de Comentarios'}
)
fig_bar.show()

<Figure: Sentimiento de Marca por Categoría de Servicio (Claro RD)>

### 7. Exportación del Dataset Procesado
Se guarda el dataset enriquecido para consumo directo del Dashboard Ejecutivo.

In [8]:
out_path = '../data/processed/youtube_claro_processed.csv'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
df_scored.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"Dataset procesado guardado exitosamente en: {out_path}")

Dataset procesado guardado exitosamente en: ../data/processed/youtube_claro_processed.csv
